# Sprint 5: Late-Fusion Policy Decision Engine & NRRP Validation

Proposal Sec 3.2.3, Sec 3.4.3 (weeks 9–10), Sec 3.6.1, and FR-05: integrate the trained **Attention U-Net** (flood hazard probability) and **GraphSAGE** (3-class structural deprivation probability vector) into the **FusionEngine** class, classify every parcel across **Kibera, Mathare, and Mukuru** into one of three governance categories, and validate the parcel outputs against the **Nairobi Rivers Regeneration Programme (NRRP)** high-risk riparian reference zones.

| Flood Hazard Regime | Low Deprivation (`0`) | Medium Deprivation (`1`) | High Deprivation (`2`) |
|---|---|---|---|
| **Low** ($p < 0.30$) | **Retain and Formalise** (`#2ecc71`) | **Retain and Upgrade** (`#f39c12`) | **Retain and Upgrade** (`#f39c12`) |
| **Moderate** ($0.30 \le p < 0.60$) | **Retain and Upgrade** (`#f39c12`) | **Retain and Upgrade** (`#f39c12`) | **Compensated Resettlement** (`#e74c3c`) |
| **High** ($p \ge 0.60$) | **Compensated Resettlement** (`#e74c3c`) | **Compensated Resettlement** (`#e74c3c`) | **Compensated Resettlement** (`#e74c3c`) |

**What this notebook does**

1. Checks out the Sprint 5 branch (or `main` once merged) and verifies the `src/models/fusion_engine.py` APIs.
2. Loads (or trains if checkpoints are absent) the Sprint 4 **GraphSAGE** and **Attention U-Net** models and ensures per-settlement graphs carry metric parcel coordinates (`data.pos`).
3. Stitches overlapping $256\times 256$ U-Net tile predictions back onto each settlement's raster grid (`predict_settlement_flood_map`) and samples parcel-level flood probabilities at each building footprint (`extract_parcel_flood_probabilities`).
4. Runs `FusionEngine.classify_parcels` across Kibera, Mathare, and Mukuru, concatenating `[p_flood, p_dep_low, p_dep_med, p_dep_high]` and applying the $3\times 3$ policy matrix.
5. Validates the three-class parcel recommendations against the NRRP high-risk riparian reference zones (or an OSM river-corridor buffer fallback) via `validate_policy_against_nrrp`, reporting **Compensated Resettlement lift** ($> 1.0$) and the **blanket-eviction avoided** governance metric.
6. Serialises the policy-classified parcel GeoJSON layer per settlement to `outputs/classifications/<settlement>_policy_parcels.geojson` (and copies to Drive) ready for Sprint 6's Folium XAI dashboard.

**Offline / Cloud Agent note.** Real three-settlement GEE rasters, Overpass pulls, and Drive checkpoints live in Colab. All `src/models/fusion_engine.py` functions are unit-tested offline against synthetic arrays and geometries (`tests/test_fusion_engine.py`).

## 0. Before you run

- **Runtime → Change runtime type → T4 GPU** recommended (inference runs on CPU too, but GPU speeds up U-Net tile stitching or fallback training if Sprint 4 checkpoints are not on Drive).
- Mount Google Drive so Sprint 4 checkpoints (`checkpoints/sprint4/`), tiled stacks (`data/tiles/`), and exported GeoTIFFs (`*_raster_stack.tif`) are accessible.
- If you have the digitized **NRRP 37 high-risk zone polygons** (GeoJSON / Shapefile), set `NRRP_GEOJSON_PATH` in cell 5; otherwise the notebook builds a 120 m buffer around OSM river/stream corridors (`waterway=river|stream|drain`) as the riparian reference fallback (same fallback used in Sprint 2 v2 section 5).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — inference and late fusion run fine on CPU.")

In [ ]:
%%bash
# Prefer the Sprint 5 PR branch while open; falls back to main after merge.
BRANCH=cursor/sprint5-late-fusion-nrrp-57ae
REPO=https://github.com/emmanuelamor5/nairobi-risk-mapping.git

if [ -d /content/nairobi-risk-mapping/.git ]; then
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || git fetch origin main
  git checkout "$BRANCH" 2>/dev/null || git checkout main
  git pull --ff-only origin "$BRANCH" 2>/dev/null || git pull --ff-only origin main
else
  git clone "$REPO" /content/nairobi-risk-mapping
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || true
  git checkout "$BRANCH" 2>/dev/null || git checkout main
fi
cd /content/nairobi-risk-mapping
git rev-parse --abbrev-ref HEAD
ls src/models/fusion_engine.py

In [ ]:
%cd /content/nairobi-risk-mapping
!pip install -q -r requirements.txt

In [ ]:
# --- Config knobs ---
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/nairobi_risk_mapping")
except ImportError:
    print("Not in Colab — artefacts stay under the repo working directory.")
    DRIVE_ROOT = Path(".")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Symlink repo data/ -> Drive so DataPipeline paths stay relative.
DATA_LINK = Path("data")
if not (DATA_LINK.is_symlink() or DATA_LINK.exists()):
    target = DRIVE_ROOT / "data"
    target.mkdir(parents=True, exist_ok=True)
    DATA_LINK.symlink_to(target)

SETTLEMENT_KEYS = ("kibera", "mathare", "mukuru")
REBUILD_GRAPHS_IF_NO_POS = True   # Rebuild graph via Overpass if saved .pt predates data.pos
TRAIN_IF_CHECKPOINT_MISSING = True

# Late-fusion policy thresholds (isolated in config / FusionEngine per proposal Sec 3.6.1)
FLOOD_LOW_THRESHOLD = 0.30        # p < 0.30 -> low flood regime
FLOOD_HIGH_THRESHOLD = 0.60       # p >= 0.60 -> high flood regime

# NRRP validation reference
NRRP_GEOJSON_PATH = None          # e.g. str(DRIVE_ROOT / "nrrp_37_high_risk_zones.geojson")
RIVER_BUFFER_M = 120.0            # OSM river-corridor fallback buffer (metres)

SPRINT4_CKPT_DIR = DRIVE_ROOT / "checkpoints" / "sprint4"
OUTPUT_CLASSIFICATIONS_DIR = DRIVE_ROOT / "outputs" / "classifications"
OUTPUT_CLASSIFICATIONS_DIR.mkdir(parents=True, exist_ok=True)
Path("outputs/classifications").mkdir(parents=True, exist_ok=True)

print("DATA ->", DATA_LINK.resolve())
print("SPRINT 4 CHECKPOINTS ->", SPRINT4_CKPT_DIR)
print("CLASSIFICATIONS OUT ->", OUTPUT_CLASSIFICATIONS_DIR)

## 1. Repo check — Sprint 5 Late-Fusion & NRRP APIs exist

In [ ]:
from src import config
from src.models.fusion_engine import (
    DEFAULT_POLICY_MATRIX,
    FusionEngine,
    aggregate_multi_settlement_fusion_reports,
    build_parcel_policy_geodataframe,
    discretize_flood_probability,
    export_settlement_policy_geojson,
    extract_parcel_flood_probabilities,
    flag_parcels_in_reference_zone,
    predict_settlement_flood_map,
    validate_policy_against_nrrp,
)
from src.models.multi_settlement import (
    combine_settlement_graphs,
    load_multi_settlement_manifests,
    load_settlement_graphs,
    multi_settlement_spatial_holdout,
    prepare_multi_settlement_data,
)
from src.models.graphsage_model import GraphSAGEModel
from src.models.attention_unet_model import AttentionUNetModel
from src.models.dataset import DEFAULT_LAYOUT, EVENT_S1_LAYOUT

engine = FusionEngine(
    flood_low_threshold=FLOOD_LOW_THRESHOLD,
    flood_high_threshold=FLOOD_HIGH_THRESHOLD,
)
assert engine.policy_matrix.shape == (3, 3)
assert tuple(config.POLICY_CLASS_NAMES) == (
    "Retain and Formalise",
    "Retain and Upgrade",
    "Compensated Resettlement",
)
print("Policy classes:", config.POLICY_CLASS_NAMES)
print("Policy colors:", config.POLICY_CLASS_COLORS)
print("3x3 policy matrix (rows=flood [low,mod,high], cols=deprivation [low,med,high]):")
print(engine.policy_matrix)

## 2. Load settlement graphs & GraphSAGE deprivation probabilities

Loads `data/graphs/{settlement}_graph.pt` for Kibera, Mathare, and Mukuru. If an older `.pt` file from Sprint 3/4 does not yet store metric building centroid coordinates (`data.pos`, added in Sprint 5 so `from_networkx` does not overwrite the `y` coordinate with `data.y` labels) and `REBUILD_GRAPHS_IF_NO_POS=True`, `DataPipeline.acquire_graph_data()` refreshes the graph and saves `data/graphs/{settlement}_buildings.geojson`.

In [ ]:
import geopandas as gpd
import numpy as np
import torch
from src.data_pipeline import DataPipeline

graphs_dir = Path("data/graphs")
graphs_dir.mkdir(parents=True, exist_ok=True)

graphs = load_settlement_graphs(graphs_dir, settlement_keys=SETTLEMENT_KEYS)

for key in SETTLEMENT_KEYS:
    needs_rebuild = (
        key not in graphs
        or not hasattr(graphs[key], "pos")
        or graphs[key].pos is None
    )
    if needs_rebuild and REBUILD_GRAPHS_IF_NO_POS:
        print(f"Rebuilding graph for {key} to attach metric parcel coordinates (data.pos)...")
        pipe = DataPipeline(settlement_key=key)
        g_data, integrity = pipe.acquire_graph_data()
        graphs[key] = g_data
        print(f"  {key}: nodes={g_data.num_nodes}, pos={tuple(g_data.pos.shape)}")

if not graphs:
    raise RuntimeError("No settlement graphs available. Set REBUILD_GRAPHS_IF_NO_POS=True in Colab.")

combined = prepare_multi_settlement_data(graphs, seed=42)
gs_model = GraphSAGEModel(
    in_channels=combined.x.size(1),
    hidden_channels=64,
    num_classes=config.DEPRIVATION_NUM_CLASSES,
    dropout=0.5,
)

gs_ckpt = SPRINT4_CKPT_DIR / "graphsage" / "graphsage_best.pt"
if gs_ckpt.exists():
    gs_model.load_checkpoint(str(gs_ckpt))
    print("Loaded Sprint 4 GraphSAGE checkpoint from:", gs_ckpt)
elif TRAIN_IF_CHECKPOINT_MISSING:
    print("Sprint 4 GraphSAGE checkpoint not found — training on combined graph...")
    gs_res = gs_model.fit(
        combined,
        epochs=100,
        checkpoint_dir=str(SPRINT4_CKPT_DIR / "graphsage"),
        early_stopping_patience=25,
        seed=42,
    )
    print(f"Trained GraphSAGE: best_val_macro_f1={gs_res['best_val_macro_f1']:.4f}")

# Predict 3-class deprivation probability vectors per settlement
deprivation_probs_by_settlement = {}
for key, g in graphs.items():
    probs = gs_model.predict_proba(g).numpy()
    deprivation_probs_by_settlement[key] = probs
    pred_cls = probs.argmax(axis=1)
    counts = {config.DEPRIVATION_CLASS_NAMES[i]: int((pred_cls == i).sum()) for i in range(3)}
    print(f"{key:8s} | parcels={len(probs):6d} | predicted deprivation classes={counts}")

## 3. Load Attention U-Net & stitch settlement flood probability surfaces

Loads the Sprint 4 Attention U-Net checkpoint (`checkpoints/sprint4/attention_unet/attention_unet_best.pt`, or trains if missing), runs inference over each settlement's un-augmented `*_orig` tiles (`predict_settlement_flood_map`), and averages overlapping 50 px strips to produce a continuous flood probability map aligned with `{settlement}_raster_stack.tif`.

In [ ]:
import glob
import os
import rasterio

tiles_root = Path("data/tiles")
manifests = load_multi_settlement_manifests(
    tiles_root, settlement_keys=SETTLEMENT_KEYS, split="train", require_all=False,
)
print("Tile manifests found:", {k: len(v) for k, v in manifests.items()})

# Determine stack layout (12-band EVENT_S1_LAYOUT vs 10-band DEFAULT_LAYOUT)
layout = EVENT_S1_LAYOUT
if manifests:
    first_key = next(iter(manifests))
    sample_tile = np.load(manifests[first_key][0]["path"])
    if sample_tile.shape[0] == len(DEFAULT_LAYOUT.band_names):
        layout = DEFAULT_LAYOUT
print("Using layout:", layout.band_names, "| in_channels=", layout.in_channels)

unet = AttentionUNetModel(layout=layout, dropout=0.3)
unet_ckpt = SPRINT4_CKPT_DIR / "attention_unet" / "attention_unet_best.pt"
if unet_ckpt.exists():
    unet.load_checkpoint(str(unet_ckpt))
    print("Loaded Sprint 4 Attention U-Net checkpoint from:", unet_ckpt)
elif len(manifests) >= 2 and TRAIN_IF_CHECKPOINT_MISSING:
    print("Sprint 4 U-Net checkpoint not found — training on pooled settlement tiles...")
    split = multi_settlement_spatial_holdout(manifests, holdout_fraction=0.25, seed=42)
    u_res = unet.fit(
        split["train"], split["val"],
        epochs=40, batch_size=4, learning_rate=1e-3, weight_decay=1e-4,
        early_stopping_patience=12,
        checkpoint_dir=str(SPRINT4_CKPT_DIR / "attention_unet"),
    )
    if unet_ckpt.exists():
        unet.load_checkpoint(str(unet_ckpt))
    print(f"Trained U-Net: best_val_iou={u_res['best_val_iou']:.4f}")


def locate_settlement_stack(settlement_key: str, expected_bands: int = None):
    """Locate the newest exported GeoTIFF for ``settlement_key`` on Drive or data/."""
    search_patterns = [
        str(DRIVE_ROOT / f"{settlement_key}_raster_stack*.tif"),
        f"data/raster_stacks/{settlement_key}_raster_stack*.tif",
    ]
    candidates = []
    for pat in search_patterns:
        candidates.extend(glob.glob(pat))
    candidates = sorted(set(candidates), key=os.path.getmtime, reverse=True)
    for p in candidates:
        if expected_bands is None:
            return p
        with rasterio.open(p) as src:
            if src.count == expected_bands:
                return p
    return candidates[0] if candidates else None


flood_maps_by_settlement = {}
for key in graphs:
    stack_path = locate_settlement_stack(key, expected_bands=len(layout.band_names))
    if stack_path and key in manifests:
        with rasterio.open(stack_path) as src:
            raster_shape = (src.height, src.width)
            transform = src.transform
            raster_crs = src.crs
            bounds = src.bounds
        stitched = predict_settlement_flood_map(
            unet, manifests[key], raster_shape=raster_shape, layout=layout,
        )
        flood_maps_by_settlement[key] = {
            "prob_map": stitched["prob_map"],
            "coverage_mask": stitched["coverage_mask"],
            "transform": transform,
            "crs": raster_crs,
            "bounds": bounds,
            "stack_path": stack_path,
        }
        cov_pct = stitched["coverage_mask"].mean() * 100.0
        mean_p = float(stitched["prob_map"][stitched["coverage_mask"]].mean())
        print(f"{key:8s} | stitched {raster_shape[1]}x{raster_shape[0]} raster | "
              f"tile coverage={cov_pct:.1f}% | mean flood prob={mean_p:.4f}")
    else:
        print(f"WARNING: {key} stack or tile manifest missing on disk (stack={stack_path}).")

## 4. Late-Fusion Policy Classification & NRRP High-Risk Zone Validation

For each settlement:
1. Sample the stitched U-Net flood probability map at each building footprint (`extract_parcel_flood_probabilities`).
2. Rasterise the NRRP high-risk zone polygons (or the 120 m OSM river-corridor buffer fallback) onto the settlement grid and flag which parcels lie inside the high-risk riparian zone (`flag_parcels_in_reference_zone`).
3. Run `FusionEngine.classify_parcels(flood_probs, deprivation_probs)` to assign each parcel to **Retain and Formalise**, **Retain and Upgrade**, or **Compensated Resettlement**.
4. Validate the policy layer against the NRRP reference (`validate_policy_against_nrrp`) and export `outputs/classifications/<settlement>_policy_parcels.geojson`.

In [ ]:
import osmnx as ox
from shapely.geometry import box
from src.utils import label_validation

per_settlement_fusion = {}
parcel_gdfs = {}

for key, g in graphs.items():
    print("=" * 70)
    print(f"LATE FUSION & NRRP VALIDATION: {key.upper()}")
    print("=" * 70)

    dep_probs = deprivation_probs_by_settlement[key]
    n_parcels = len(dep_probs)

    # Load cached building polygons if available; otherwise use data.pos (EPSG:32737)
    buildings_path = graphs_dir / f"{key}_buildings.geojson"
    buildings_gdf = gpd.read_file(buildings_path) if buildings_path.exists() else None
    if buildings_gdf is not None and len(buildings_gdf) != n_parcels:
        buildings_gdf = None
    node_pos = g.pos.cpu().numpy() if hasattr(g, "pos") and g.pos is not None else None

    if key in flood_maps_by_settlement:
        finfo = flood_maps_by_settlement[key]
        flood_probs = extract_parcel_flood_probabilities(
            prob_map=finfo["prob_map"],
            transform=finfo["transform"],
            raster_crs=finfo["crs"],
            buildings_gdf=buildings_gdf,
            node_positions_metric=node_pos,
        )

        # Build NRRP / OSM river-corridor reference mask on the same raster grid
        out_shape = finfo["prob_map"].shape
        if NRRP_GEOJSON_PATH and Path(NRRP_GEOJSON_PATH).exists():
            nrrp_gdf = gpd.read_file(NRRP_GEOJSON_PATH)
            ref_mask = label_validation.rasterize_reference(
                nrrp_gdf, finfo["transform"], out_shape, finfo["crs"],
            )
            ref_source = f"NRRP polygons ({NRRP_GEOJSON_PATH})"
        else:
            b = finfo["bounds"]
            aoi = gpd.GeoDataFrame(
                geometry=[box(b.left, b.bottom, b.right, b.top)], crs=finfo["crs"],
            ).to_crs(config.CRS_WGS84)
            waterways = ox.features_from_polygon(
                aoi.geometry.iloc[0],
                tags={"waterway": ["river", "stream", "drain"]},
            )
            waterways = waterways[waterways.geometry.type.isin(["LineString", "MultiLineString"])]
            ref_mask = label_validation.river_corridor_reference(
                waterways.to_crs(finfo["crs"]),
                finfo["transform"],
                out_shape,
                finfo["crs"],
                buffer_m=RIVER_BUFFER_M,
            )
            ref_source = f"OSM river corridor ({RIVER_BUFFER_M:.0f}m buffer)"

        in_nrrp = flag_parcels_in_reference_zone(
            reference_mask=ref_mask,
            transform=finfo["transform"],
            raster_crs=finfo["crs"],
            buildings_gdf=buildings_gdf,
            node_positions_metric=node_pos,
        )
    else:
        raise RuntimeError(
            f"Flood raster stack missing for '{key}'. Ensure {key}_raster_stack.tif "
            f"and data/tiles/{key}/manifest_train.json are present on Drive."
        )

    cls_result = engine.classify_parcels(flood_probs, dep_probs)
    summary = engine.summarize_classifications(cls_result)
    nrrp_report = validate_policy_against_nrrp(
        policy_class=cls_result["policy_class"],
        in_nrrp_zone=in_nrrp,
        flood_prob=cls_result["flood_prob"],
        deprivation_class=cls_result["deprivation_class"],
    )

    per_settlement_fusion[key] = {
        "classification": cls_result,
        "summary": summary,
        "in_nrrp_zone": in_nrrp,
        "nrrp_report": nrrp_report,
        "ref_source": ref_source,
    }

    # Build and export the parcel policy GeoJSON layer (proposal Sec 4.4.1)
    node_ids = g.node_id.cpu().numpy() if hasattr(g, "node_id") and g.node_id is not None else None
    parcel_gdf = build_parcel_policy_geodataframe(
        settlement_key=key,
        classification_result=cls_result,
        buildings_gdf=buildings_gdf,
        node_positions_metric=node_pos,
        node_ids=node_ids,
        in_nrrp_zone=in_nrrp,
    )
    parcel_gdfs[key] = parcel_gdf
    local_geojson = export_settlement_policy_geojson(parcel_gdf, key, output_dir="outputs/classifications")
    drive_geojson = export_settlement_policy_geojson(parcel_gdf, key, output_dir=OUTPUT_CLASSIFICATIONS_DIR)

    print(f"Reference zone: {ref_source} | parcels in zone: {nrrp_report['nrrp_zone_parcels']}/{n_parcels} "
          f"({nrrp_report['nrrp_zone_fraction']*100:.1f}%)")
    print("Policy breakdown:")
    for pname in config.POLICY_CLASS_NAMES:
        cnt = summary["counts"][pname]
        frac = summary["fractions"][pname] * 100.0
        mfp = summary["mean_flood_prob_by_policy"][pname]
        mcf = summary["mean_confidence_by_policy"][pname]
        print(f"  {pname:26s}: {cnt:6d} ({frac:5.1f}%) | mean_flood_p={mfp:.3f} | mean_conf={mcf:.3f}")
    print(f"NRRP validation:")
    print(f"  Compensated Resettlement lift in NRRP zone : {nrrp_report['resettlement_lift']:.2f}x "
          f"(meets > 1.0: {nrrp_report['meets_lift_target']})")
    print(f"  Intervention (Upgrade+Resettle) lift       : {nrrp_report['intervention_lift']:.2f}x")
    print(f"  Mean flood prob in NRRP vs outside         : "
          f"{nrrp_report['mean_flood_prob_in_nrrp']:.3f} vs {nrrp_report['mean_flood_prob_outside_nrrp']:.3f}")
    print(f"  Blanket eviction avoided in riparian zone  : "
          f"{nrrp_report['retained_in_nrrp_count']}/{nrrp_report['nrrp_zone_parcels']} "
          f"({nrrp_report['blanket_eviction_avoided_fraction']*100:.1f}% retained via Formalise/Upgrade)")
    print(f"  Exported GeoJSON -> {drive_geojson}")

## 5. Pooled three-settlement summary & visual diagnostics

In [ ]:
import json
import matplotlib.pyplot as plt

pooled_report = aggregate_multi_settlement_fusion_reports(
    per_settlement_fusion, engine=engine,
)

print("=" * 70)
print("SPRINT 5 POOLED THREE-SETTLEMENT LATE-FUSION SUMMARY")
print("=" * 70)
ps = pooled_report["pooled_summary"]
pn = pooled_report["pooled_nrrp"]
print(f"Total parcels classified across { list(per_settlement_fusion) }: {ps['total_parcels']}")
for pname in config.POLICY_CLASS_NAMES:
    print(f"  {pname:26s}: {ps['counts'][pname]:6d} ({ps['fractions'][pname]*100:5.1f}%)")

if pn is not None:
    print("\nPooled NRRP High-Risk Zone Validation:")
    print(f"  Parcels inside NRRP / riparian zone        : {pn['nrrp_zone_parcels']} ({pn['nrrp_zone_fraction']*100:.1f}%)")
    print(f"  Compensated Resettlement lift              : {pn['resettlement_lift']:.2f}x "
          f"(meets > {pn['lift_target']}: {pn['meets_lift_target']})")
    print(f"  All settlements individually meet lift bar : {pooled_report['all_settlements_meet_lift_target']}")
    print(f"  Blanket eviction avoided in riparian zone  : "
          f"{pn['retained_in_nrrp_count']}/{pn['nrrp_zone_parcels']} "
          f"({pn['blanket_eviction_avoided_fraction']*100:.1f}%)")

# Persist Sprint 5 summary JSON alongside the GeoJSON layers
summary_path = OUTPUT_CLASSIFICATIONS_DIR / "sprint5_fusion_nrrp_summary.json"
with open(summary_path, "w") as f:
    json.dump(pooled_report, f, indent=2, default=str)
print("\nSaved Sprint 5 summary JSON to:", summary_path)

In [ ]:
# Plot per-settlement policy distribution & inside-vs-outside NRRP breakdown
settlements = list(per_settlement_fusion.keys())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(settlements))
width = 0.25
for idx, (pname, color) in enumerate(zip(config.POLICY_CLASS_NAMES, config.POLICY_CLASS_COLORS)):
    shares = [per_settlement_fusion[k]["summary"]["fractions"][pname] * 100.0 for k in settlements]
    axes[0].bar(x + (idx - 1) * width, shares, width, label=pname, color=color)
axes[0].set_xticks(x)
axes[0].set_xticklabels([k.capitalize() for k in settlements])
axes[0].set_ylabel("Share of parcels (%)")
axes[0].set_title("Parcel Policy Recommendations by Settlement")
axes[0].legend()

if pn is not None:
    zones = ["Inside NRRP / Riparian Zone", "Outside High-Risk Zone"]
    zx = np.arange(len(zones))
    for idx, (pname, color) in enumerate(zip(config.POLICY_CLASS_NAMES, config.POLICY_CLASS_COLORS)):
        vals = [
            pn["in_nrrp_policy_fractions"][pname] * 100.0,
            pn["outside_nrrp_policy_fractions"][pname] * 100.0,
        ]
        axes[1].bar(zx + (idx - 1) * width, vals, width, label=pname, color=color)
    axes[1].set_xticks(zx)
    axes[1].set_xticklabels(zones)
    axes[1].set_ylabel("Share of zone parcels (%)")
    axes[1].set_title(f"NRRP Zone Policy Concentration (Resettlement Lift = {pn['resettlement_lift']:.2f}x)")
    axes[1].legend()

plt.tight_layout()
plt.show()

## Sprint 5 exit checklist

- [ ] `FusionEngine` concatenates Attention U-Net flood probabilities and GraphSAGE deprivation probability vectors `[p_flood, p_low, p_med, p_high]` and applies the $3\times 3$ rule-based policy matrix.
- [ ] Overlapping $256\times 256$ U-Net tiles stitched per settlement and sampled at building footprints.
- [ ] Three-class parcel outputs validated against NRRP / riparian high-risk reference zones (`resettlement_lift > 1.0`, plus the blanket-eviction avoided share inside the riparian buffer).
- [ ] Policy-classified parcel GeoJSON layers exported for **Kibera**, **Mathare**, and **Mukuru** under `outputs/classifications/<settlement>_policy_parcels.geojson`.

Next: **Sprint 6** — Captum Grad-CAM spatial attention overlays, SHAP feature attribution charts, and the standalone interactive Folium XAI dashboard (`DashboardRenderer`).